# The Panel and Portfolio Mathematics
## 🎯 Learning Objectives

By the end of today you will be able to:

1. **Read a stacked panel** — why stock data is stored long, not wide, and why
   `groupby('date')` is the workhorse operation of this entire course
2. **Explain what CRSP is** and why delisted companies have to be in your data
3. **Compute a portfolio return** as a weighted average, `w′r`, in one line
4. **Distinguish value- from equal-weighting** and say what each one is a bet on
5. **Weight a portfolio without look-ahead** — using only information you had
   at the time
6. **Rebuild the US stock market from scratch** and check it against the
   official series

## 📋 Today's Plan

1. [Where this data comes from](#data)
2. [The stacked panel](#panel)
3. [What returns actually look like](#distributions)
4. [Pitfall checklist](#pitfalls)
5. [Portfolio weights](#weights)
6. [🔄 Live Demo: portfolio returns, `w′r`](#demo1)
7. [Value- vs equal-weighting](#vwew)
8. [🛠️ Hands-On: which one is bigger, and why?](#ho1)
9. [🎯 Challenge: rebuild the market](#challenge)
10. [Key takeaways](#takeaways)

---

## 🛠️ Setup

In [ ]:
#@title Setup — run this first
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [11, 4.5]
plt.rcParams['font.size'] = 11
import warnings; warnings.filterwarnings('ignore')
print("✅ Ready")

---

## 1. Where This Data Comes From <a id="data"></a>

Every number you will use this semester came from somewhere, and the choices
made in building it are already baked in before you write a line of code.

### CRSP

The **Center for Research in Security Prices** at Chicago maintains the standard
academic database of US stock returns — every US-listed common stock, monthly
and daily, back to 1926. It is the source for essentially every published paper
on US equity returns, and it is what your panel is built from.

Two things make CRSP the standard rather than, say, Yahoo Finance:

**It includes companies that no longer exist.** Enron is in CRSP. Lehman is in
CRSP. Every company that went bankrupt, got acquired, or was delisted for
failing to meet listing standards is still there, with its returns, right up to
the end.

> **⚠️ Caution: survivorship bias**
>
> Build a dataset from today's listed companies and you have automatically
> excluded every failure. Backtests on such data look wonderful and mean
> nothing. If you pull "all S&P 500 stocks" from a free API today and test a
> strategy back to 1990, you are testing "what if I had known in 1990 which
> companies would still be in the index in 2026."

**It handles the delisting return.** When a company is delisted, there is one
final return — sometimes −100%, sometimes a buyout premium — that is recorded
separately from the normal monthly series. Ignore it and you silently drop the
worst month of every failed company.

> **📌 Your panel already has this fixed.** The build merged 1,229 delisting
> returns into the monthly series. You'll see the code at the end of Lecture 3.

### Compustat

CRSP has prices and returns. It has no idea what a company *earns* or *owns*.
Accounting data — book value, earnings, assets, debt — comes from **Compustat**,
and joining the two is one of the standard chores of empirical finance. We'll
use pre-joined signals rather than doing that merge by hand.

---

## 2. The Stacked Panel <a id="panel"></a>

In [ ]:
URL = ("https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/panel_backbone_1980_2000.parquet")
panel = pd.read_parquet(URL)

print(f"{len(panel):,} rows")
print(f"{panel.permno.nunique():,} stocks over {panel.date.nunique()} months "
      f"({panel.date.min().date()} to {panel.date.max().date()})")
print(f"average of {len(panel)/panel.date.nunique():.0f} stocks per month\n")
panel.head(5)

### Long, not wide

Notice the shape: one row per **stock-month**, not one column per stock. This is
a *stacked* or *long* panel, and it is how essentially all cross-sectional
finance data is stored.

Why not a wide matrix, dates down and tickers across? A rectangular frame needs
two coordinates — row and column — to identify one asset's return on one date.
That is easier to eyeball, and it breaks twice over here.

It breaks on **variables**: one wide frame holds exactly one quantity, so you
would need a separate frame for returns, another for market equity, another for
each signal. By the time you have thirty characteristics you have thirty frames
to keep aligned.

It breaks on **assets**: the set of stocks changes every month — companies IPO,
get acquired, go bankrupt — so the matrix would be mostly empty and would need
reshaping every time the universe changed.

Stacked, you need two coordinates in *columns* — `date` and `permno` — to
identify a firm-month, and a third, the column name, to pick the variable. Every
new signal is one more column, not one more dataframe.

> **🐍 Python Insight: `groupby('date')`**
>
> Long format means "do something to each month's cross-section" is
> `df.groupby('date')`. That single operation — split by date, compute across
> stocks, recombine — is the workhorse of this entire course. Portfolio returns,
> sorts, breakpoints, factor construction: all of them are `groupby('date')`.

### The columns

| Column | What it is |
|---|---|
| `permno` | CRSP's permanent stock identifier — never reused, unlike tickers |
| `date` | Month end |
| `ret` | Total return **during** that month (dividends included) |
| `me` | Market equity (price × shares outstanding), \$thousands |
| `prc` | Price. **Can be negative** — CRSP flags a bid/ask midpoint that way |
| `exchcd` | 1 = NYSE, 2 = AMEX, 3 = NASDAQ |
| `shrcd` | Share code; 10/11 are ordinary common shares |

### Tell me three basic questions that we can answer with this data set?

In [ ]:
## your questions and your answer here

---

## 3. What Returns Actually Look Like <a id="distributions"></a>

Almost every formula you will meet this semester — Sharpe ratios, mean-variance
optimization, confidence intervals — behaves as though returns were normal.
Before building anything, look at whether they are.

Start with one company you know.

In [ ]:
from scipy import stats
ge = panel[panel.permno == 12060].set_index('date')['ret'].dropna()      # GE, 252 months

print(f"General Electric, {len(ge)} monthly returns, 1980-2000\n")
print(f"   volatility        {ge.std()*np.sqrt(12):>7.1%} /yr")
print(f"   skewness          {stats.skew(ge):>7.2f}      (normal: 0)")
print(f"   excess kurtosis   {stats.kurtosis(ge):>7.2f}      (normal: 0)")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
z = (ge - ge.mean()) / ge.std()
ax.hist(z, bins=40, density=True, alpha=0.65, color='steelblue', label='GE')
t = np.linspace(-4, 4, 300)
ax.plot(t, stats.norm.pdf(t), 'k--', lw=1.6, label='Normal')
ax.set_xlabel('standard deviations from the mean'); ax.legend()
ax.set_title('One company, 252 months', fontweight='bold')
plt.tight_layout(); plt.show()

### But GE is one company

That histogram is close to a bell curve — and it would be a mistake to conclude
anything from it. GE is a large, old, heavily-followed firm; there are 6,000
stocks in this panel and most look nothing like it.

So measure **every** stock on its own returns, then average across stocks.

In [ ]:
# One row per stock, each measured on its own history. 10-year minimum so the
# moments mean something.
g   = panel.dropna(subset=['ret']).groupby('permno')['ret']
n   = g.size()
big = n[n >= 120].index
per = pd.DataFrame({'vol':  g.std()[big] * np.sqrt(12),
                    'skew': g.apply(stats.skew)[big],
                    'kurt': g.apply(stats.kurtosis)[big]})

print(f"{len(per):,} stocks with 10+ years of history\n")
print(f"{'':20s}{'average':>10s}{'median':>10s}")
print("-"*40)
print(f"{'volatility /yr':20s}{per['vol'].mean():>10.0%}{per['vol'].median():>10.0%}")
print(f"{'skewness':20s}{per['skew'].mean():>10.2f}{per['skew'].median():>10.2f}")
print(f"{'excess kurtosis':20s}{per['kurt'].mean():>10.2f}{per['kurt'].median():>10.2f}")

print(f"\n   {(per['skew']>0).mean():.0%} of stocks are positively skewed")
print(f"   {(per['kurt']>0).mean():.0%} have fatter tails than a normal")

The average stock runs **52% volatility a year**, is **positively skewed**, and
has **fatter tails than a normal** — 99% of them do. GE, at 22% volatility and
essentially zero excess kurtosis, is unusually well-behaved.

Now the portfolio that holds all of them.

In [ ]:
import pandas_datareader.data as web
ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01', end='2000-12-31')[0] / 100
ff.index = pd.to_datetime(ff.index.to_timestamp()) + pd.offsets.MonthEnd(0)
mkt = (ff['Mkt-RF'] + ff['RF']).rename('ff_mkt')



print(f"The value-weighted market, {len(mkt)} months\n")
print(f"{'':20s}{'average stock':>15s}{'the market':>13s}")
print("-"*48)
print(f"{'volatility /yr':20s}{per['vol'].mean():>15.0%}{mkt.std()*np.sqrt(12):>13.1%}")
print(f"{'skewness':20s}{per['skew'].mean():>15.2f}{stats.skew(mkt):>13.2f}")
print(f"{'excess kurtosis':20s}{per['kurt'].mean():>15.2f}{stats.kurtosis(mkt):>13.2f}")
# The Fama-French series is already dated by the month the return was EARNED,
# so no shift is needed here.
worst = mkt.idxmin().strftime('%B %Y')
print(f"\n   worst month: {mkt.min():.1%}  (earned in {worst})")

fig, ax = plt.subplots(figsize=(7.5, 3.4))
zm = (mkt - mkt.mean()) / mkt.std()
ax.hist(zm, bins=40, density=True, alpha=0.65, color='darkorange', label='Market')
ax.plot(t, stats.norm.pdf(t), 'k--', lw=1.6, label='Normal')
ax.set_xlabel('standard deviations from the mean'); ax.legend()
ax.set_title(f'The market, {len(mkt)} months', fontweight='bold')
plt.tight_layout(); plt.show()

### Three stylized facts

**1. Almost every stock has fatter tails than a normal.** 99% of them. The
average excess kurtosis is 6.2, and the average stock swings 52% a year — more
than three times the market.

**2. Individual stocks are skewed right; the market is skewed left.** A single
stock can rise 2,400% and can only fall 100%, so 89% of them lean right. The
market leans the other way, at **−0.83** — crashes are the thing that happens to
everything at once. The worst month here is October 1987, at **−22.6%**.

**3. Diversification shrinks the size of the moves, not the shape of the tail.**
Holding all 6,000 takes volatility from 52% to **15.5%** — an enormous
reduction, and essentially the whole reason to hold a portfolio. It does not
remove the crash: the market's excess kurtosis is still **3.3**.

> **🤔 Two questions to sit with **
>
> **Why doesn't the volatility go to zero?** There are about **5,000** stocks in
> here, each swinging around 52% a year. If they moved independently of one
> another, averaging 6,000 of them would leave you with 52% ÷ √6000 ≈ **0.7%** a
> year. The actual market runs **15.5%** — more than **twenty times** that. So
> something is stopping the averaging from working. What?
>
> **And how can 89% of stocks be skewed right while the market is skewed left?**
> Average 5,000 right-leaning things and you would expect a right-leaning
> average. You get the opposite sign. Averaging clearly does not do to skewness
> what it does to volatility.
>
> Both questions have the same one-word answer, and you already have the data to
> find it. Hold them — the first is the whole of Lecture 12, and the second is
> why risk models exist at all.

> **⚠️ Caution: what this costs you later**
>
> The Sharpe ratio summarizes a strategy with a mean and a standard deviation,
> which is a complete description **only** if returns are normal. Two strategies
> with identical Sharpe ratios can have very different odds of a catastrophic
> month, and Sharpe cannot see the difference.

> **🐍 Python Insight: the −100% returns are real**
>
> `panel['ret'].min()` is exactly −1.0. Those are delisting returns — companies
> that went to zero, kept in the data on purpose. A dataset without them would
> look better and be wrong.

---

## 4. Portfolio Weights <a id="weights"></a>

A portfolio is a vector of weights $w = (w_1, \dots, w_N)$, where $w_i$ is the
fraction of your capital in stock $i$.

$$\sum_{i=1}^{N} w_i = 1$$

That constraint says you invested all your money and no more. Three cases worth
naming:

| Weights | Meaning |
|---|---|
| $w_i = 1/N$ | **Equal-weighted** — same dollars in every stock |
| $w_i = ME_i / \sum_j ME_j$ | **Value-weighted** — proportional to company size |
| $\sum_i w_i = 0$ | **Long-short, self-financed** — the excess return from Lecture 1 |

That last row is the callback. When you computed $r - r^f$ last week you were
describing a portfolio with weights $(+1, -1)$ on the stock and the risk-free
asset. They sum to zero. It costs nothing to enter, so what it returns is a
spread, not a return on capital.

> **💡 Key Insight**
>
> Value-weighting is not a choice about which stocks you like. It is the only
> weighting that *everyone can hold at once*. If every investor tried to hold an
> equal-weighted portfolio, there would not be enough shares of the small
> companies to go around. The value-weighted portfolio **is** the market.

### And it costs nothing to maintain

Value weights have a property no other scheme has: **they rebalance themselves.**

If a stock doubles, its market cap doubles, and its correct weight doubles — but
so did the value of your holding. You trade nothing. Every other weighting
scheme drifts away from its target as prices move and has to be traded back.

That is the entire reason index funds work, and why the first one could charge
almost nothing. Hold this thought for Lecture 11, when we put a price on
turnover.

There is a second advantage, and it is the one that will keep coming back. By
buying in proportion to market cap you **never have to build a large position in
a tiny stock**. Equal weighting does the opposite: it forces the same dollars
into the 6,000th-largest company as into General Electric. Those portfolios are
very hard to trade, and the alphas people report from them are mostly a mirage
unless the universe is restricted to large caps.

> **📌 Remember**
>
> Value-weighted → zero turnover from price moves, and no position you can't get
> out of.
> Equal-weighted → you must sell winners and buy losers every single month, in
> the names least able to absorb it.

---

## 🔄 Live Demo: Portfolio Returns <a id="demo1"></a>

We want, for each month, the return of a portfolio holding every stock in the
market weighted by size. That is the US stock market, and you are about to build
it from 1.5 million rows.

> **🤔 Before anyone writes anything.** Say the request out loud, precisely
> enough that someone who has never seen this panel could implement it without
> asking you a question. What do they need to be told?

Three things, and only one of them is obvious:

1. Weight by `me` 
2. Get the timing right
3. Whether you drop missing values *before* or *after* forming the weights


In [ ]:
# === YOUR TURN ===
# Write the prompt first. Then paste what the AI gives you underneath.

MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below this line ----


### Timing is everything 

`ret` is the return **during** month *t*. `me` is market equity at the **end** of
month *t*. If you weight by `me` at *t* and then claim the return `ret` at *t*,
you have used end-of-month information to earn a return that was already
happening. That is **look-ahead bias**, and it will make almost any strategy
look brilliant.

> **📌 Remember: the rule for every portfolio in this course**
>
> Form weights from information available at the end of month *t*. Earn
>  the return over month *t+1*. Nothing else.

In [ ]:
# See it directly for one stock
ge = panel[panel.permno == 12060].head(4)[['date', 'ret', 'me']]
print(ge.to_string(index=False))
print("You decide at the end of January; you earn February.")

print("What do you have to do use me in a trading strategy?")

ge['me_l1']=ge['me'].shift(1)

print(ge.to_string(index=False))

### build it yourself for the entire data set

The panel gives you `ret` — the return *this* month. To sort on a signal you
need next month's return, so you have to build that column yourself. There is a
trap in the way.

> **🤔 The question.** The panel is **stacked** — every stock's history, end to
> end, in one column. You want, for each row, that stock's return next month.
> What do you have to tell an AI so it doesn't walk off the end of one company
> and into the beginning of the next?


In [ ]:
# === YOUR TURN ===
MY_PROMPT = """
                                    ← write your prompt here
"""

# ---- paste the AI's code below. Call the result `my_fwd`. ----


In [ ]:
#@title 🔒 Check — run after you've pasted yours
# Both candidate implementations, scored against a reference we build here.
p = panel.sort_values(['permno','date']).reset_index(drop=True)

# The reference: next month's return, but only when the next row really is the
# next calendar month — never splicing across a gap in a stock's history.
nxt = p.groupby('permno')['date'].shift(-1)
ok  = nxt == (p['date'] + pd.offsets.MonthEnd(1))
ref = p.groupby('permno')['ret'].shift(-1).where(ok)
cand = {'implementation 1'                  : p['ret'].shift(-1),
        'implementation 2'                      : p.groupby('permno')['ret'].shift(-1)}
try:    cand['your my_fwd'] = pd.Series(my_fwd).reset_index(drop=True)
except NameError: pass

print(f"{'implementation':40s}{'rows wrong':>12s}")
print("-"*52)
for k, v in cand.items():
    a, b = v.round(6), ref.round(6)
    print(f"{k:40s}{int(((a!=b) & ~(a.isna() & b.isna())).sum()):>12,}")
print(f"\n{p.permno.nunique():,} stocks -> {p.permno.nunique():,} places to leak.")

### Two answers, 14× apart

`shift(-1)` without the `groupby` is wrong on many rows. Not because it
errors — it doesn't — but because at every one of the 17,800 stock boundaries it
reaches across into the next company's first return.

Grouping first leaves **1,357** differences, and those are not mistakes: the
reference only fills a value when the next observation is the very next
*calendar* month, so it also refuses to splice across a gap in a stock's
history. That is a second, stricter rule you would have to ask for explicitly.

> **📌 Remember**
>
> ```python
> df['me_l1'] = df.groupby('permno')['me'].shift(1)   # ✅
> df['me_l1'] = df['me'].shift(1)                     # ❌ crosses stocks
> ```
>
> 19,156 plausible numbers scattered through 1.5 million rows. No summary
> statistic will show you they are there.


In [ ]:
#@title 🔒 Check — run after you've pasted yours
# Two implementations of "the value-weighted market return". One is right.
panel['me_l1']=panel.groupby('permno')['me'].shift(1)
d = panel.dropna(subset=['ret', 'me_l1'])
right = d.groupby('date').apply(lambda g: np.average(g['ret'], weights=g['me_l1']))

d2   = panel.dropna(subset=['ret', 'me'])
wrong = d2.groupby('date').apply(lambda g: np.average(g['ret'], weights=g['me']))

print(f"{'weights me at t-1, earn ret at t':40s}{right.mean()*12:>8.2%}/yr")
print(f"{'weights me at t,   earn ret at t':40s}{wrong.mean()*12:>8.2%}/yr")
print(f"\n  difference: {(wrong.mean()-right.mean())*12:.2%}/yr ")
print("  why such large difference?")

### What the room got

The two lines differ by **9.76 percentage points a year**. 

Ask an AI to "compute the value-weighted market return from this panel" and you
will very often get `np.average(g['ret'], weights=g['me'])`. It is elegant, it
runs, but is it a valid trading strategy? What kind of strategy it is?




### Step 3 — Validate

The weighted average above is exactly the matrix expression

$$r_p = w' r = \sum_i w_i r_i$$

Let's confirm that by doing it the explicit way for a single month — and check
the weights really do sum to one.

In [ ]:
# One month, done explicitly
m = d[d.date == '1995-06-30'].copy()
w = m['me_l1'] / m['me_l1'].sum()                  # value weights

print(f"stocks in June 1995 : {len(m):,}")
print(f"weights sum to      : {w.sum():.10f}")
print(f"largest weight      : {w.max():.4%}")
print(f"smallest weight     : {w.min():.8%}")
print(f"\nw' r  (term by term than sum)   : {(w * m['ret']).sum():.6f}")

print(f"\nw' r  (vector)   : {(w @ m['ret']):.6f}")
print(f"np.average         : {np.average(m['ret'], weights=m['me_l1']):.6f}")
print(f"vw series          : {vw.loc['1995-06-30']:.6f}")

> **🐍 Python Insight: `@` for the dot product**
>
> `(w * r).sum()` and `w @ r` compute the same thing. The `@` operator is matrix
> multiplication, and once portfolios have covariance matrices in them (Lecture
> 12) you'll want it. For now, either is fine.

> **🤔 Look at those weights before moving on.**
>
> The largest single weight is a few percent. The smallest is under a
> millionth. There are ~6,000 stocks, and the bottom half of them collectively
> barely register. Hold that thought.

---

## 🛡️ Pitfall Checklist for Building Portfolios <a id="pitfalls"></a>

| | Pitfall | What goes wrong | 🔍 How to detect |
|---|---|---|---|
| 1 | **Weighting by contemporaneous market cap** | Look-ahead — you weight by end-of-month size and claim that month's return | Are you pairing `me` at *t-1* with `ret`, at t? |
| 2 | **Weights that don't sum to 1** | Your "portfolio return" is scaled by an arbitrary constant | `w.sum()` — is it 1.0? (Or 0.0 for a long-short.) |
| 3 | **Equal-weighting by accident** | `.mean()` instead of a weighted average silently makes a small-cap bet | Did you pass a `weights=` argument at all? |
| 4 | **Dropping NaNs after forming weights** | The surviving weights no longer sum to 1 | Drop first, then compute weights |
| 5 | **Survivorship in the universe** | Delisted firms missing → returns biased up | Does the stock count fall in bad years? It should. |
| 6 | **Misaligned month-ends** | Merging `2000-01-01` against `2000-01-31` silently drops everything | Check row counts before and after a merge |
| 7 | **Double Counting** | Including ETFs | Check share code |

> **🤖 AI-Era Insight**
>
> Ask an AI to "compute the value-weighted market return from this panel" and it
> will very often write `np.average(g['ret'], weights=g['me'])` — pitfall 1,
> exactly. The code is elegant, it runs, and the resulting market return is too
> high. You have to know to lag the weights.

---

## 5. Value- vs Equal-Weighting vs ?<a id="vwew"></a>

Two portfolios, same 6,000 stocks, same months. The only difference is the
weights. They are not close to the same thing.

Equal-weighting puts the same dollars in the 3,000th-largest company as in
General Electric. Since there are far more small companies than large ones,
**an equal-weighted portfolio is overwhelmingly a bet on small stocks** — even
though nobody chose to make that bet.

---

## 🛠️ Hands-On: Which One Is Bigger, and Why? <a id="ho1"></a>

### Your task

You have `vw` and `ew`. Work out how concentrated the value-weighted portfolio
actually is: what share of the total market do the largest 10 stocks represent?
Then compare the two portfolios' returns and volatility.

> **🤔 Predict first.** Small stocks are riskier, and riskier things should earn
> more on average. So equal-weighting — which loads up on small stocks — should
> *beat* value-weighting over 20 years. Commit to yes or no before you run it.

In [ ]:
# === YOUR TURN ===
# Fill in the two blanks.

d  = __        # drop FIRST 
vw = d.__      # market cap weighted return
ew = d.__      # equal weighted return

print(f"{len(vw)} monthly observations")
print(f"VW mean {vw.mean()*12:.2%}/yr   vol {vw.std()*np.sqrt(12):.2%}")
print(f"EW mean {ew.mean()*12:.2%}/yr   vol {ew.std()*np.sqrt(12):.2%}")



last = d[d.date == d.date.max()]
top10_share =

print(f"Top 10 stocks are {top10_share:.1%} of total market cap")
print(f"...out of {len(last):,} stocks in the market that month\n")

spread = ____               
print(f"EW mean:  {ew.mean()*12:7.2%}/yr    vol {ew.std()*np.sqrt(12):6.2%}")
print(f"VW mean:  {vw.mean()*12:7.2%}/yr    vol {vw.std()*np.sqrt(12):6.2%}")
print(f"EW - VW:  {spread:+7.2%}/yr")

In [ ]:
fig, ax = plt.subplots(figsize=(11, 4.5))
ax.plot(vw.index, (1+vw).cumprod(), label='Value-weighted', linewidth=1.8)
ax.plot(ew.index, (1+ew).cumprod(), label='Equal-weighted', linewidth=1.8)
ax.set_yscale('log'); ax.set_ylabel('Growth of $1 (log scale)')
ax.set_title('Same 6,000 stocks, two weighting schemes', fontweight='bold')
ax.legend(); plt.tight_layout(); plt.show()

### What did you find?

Equal-weighting **lost** to value-weighting over 1980–2000 — by about 1.5% a
year — *and* it was substantially more volatile. More risk, less return.

That should bother you. The standard story says small stocks earn a premium.
Over these twenty years they did not.

> **💡 Key Insight**
>
> The size premium that Banz documented in 1981 largely disappeared from about
> the moment he published it. We'll look at this directly in Lecture 3, and it
> is a preview of one of the course's recurring themes: published anomalies have
> an unfortunate habit of dying once people know about them.

> **⚠️ Caution: the equal-weighted portfolio is not tradeable at scale**
>
> To hold it you would have to buy the same dollar amount of the 6,000th-largest
> US company as of GE — and many of those companies trade a few thousand dollars
> a day. Its measured return is real; your ability to capture it is not. That's
> Lecture 11.

---

## 🎯 Challenge: Rebuild the Market <a id="challenge"></a>

You just built the US stock market from 1.5 million rows of individual stock
returns. Now let's check it against the official series.

Ken French publishes the CRSP value-weighted market return, which is what
every asset-pricing paper uses as "the market." If your construction is right,
yours should match his almost exactly.

In [ ]:
# Ken French's market return (already excess of the risk-free rate, so add RF back)
import pandas_datareader.data as web
ff = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01')[0] / 100
ff.index = pd.to_datetime(ff.index.to_timestamp()) + pd.offsets.MonthEnd(0)
ff_mkt = (ff['Mkt-RF'] + ff['RF']).rename('ff_mkt')

print(ff_mkt.head(3).to_string())

### Q1 — Align and compare

Your `vw` series is indexed by the month in which you **formed** the portfolio,
but it holds the **next** month's return. Ken French's series is indexed by the
month the return was earned. Shift yours forward one month end, then join.

> **⚠️ If you skip the shift**, your correlation will be near zero and you'll
> think you did something wrong. You didn't — the series are just offset by one
> month.
>
> **📌 Required variable names:**
> ```python
> vw_mean_annual = ____   # annualized mean of your VW market return
> vw_vol_annual  = ____   # annualized volatility
> corr_vw_ff     = ____   # correlation with Ken French's series
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
vw_mean_annual = ____
vw_vol_annual  = ____
corr_vw_ff     = ____

print(f"Your VW market: {vw_mean_annual:.2%}/yr, vol {vw_vol_annual:.2%}")
print(f"Correlation with Ken French: {corr_vw_ff:.4f}")

### Q2 — The equal-weighted comparison

Report the annualized mean and volatility of the equal-weighted portfolio over
the same aligned sample.

> **📌 Required variable names:**
> ```python
> ew_mean_annual = ____
> ew_vol_annual  = ____
> ```

In [ ]:
# Your work here


# Required outputs — fill these in:
ew_mean_annual = ____
ew_vol_annual  = ____

print(f"EW: {ew_mean_annual:.2%}/yr, vol {ew_vol_annual:.2%}")
print(f"VW: {vw_mean_annual:.2%}/yr, vol {vw_vol_annual:.2%}")

### Q3 — Concentration

Report `top10_share` from the Hands-On above — the share of total market
capitalization held by the ten largest stocks in the final month.

> **📌 Required variable name:** `top10_share` *(already computed — just make
> sure it's still defined)*

### Q4 — The memo

> **📝 Your task — maximum 5 sentences**
>
> You built two portfolios from identical data and identical stocks. The
> equal-weighted one earned less with more volatility. Explain why the two
> differ so much, and argue for which of the two deserves to be called "the
> market." Mention one reason the equal-weighted return might overstate what an
> investor could actually have earned.

In [ ]:
MEMO = """
Write your memo here. Don't delete the surrounding triple quotes.
"""
print(MEMO)

---

## 📤 Submission <a id="submit"></a>

In [ ]:
# === 📤 SUBMISSION CELL — Run this last ===
import json, base64, hashlib, datetime as dt

required = ["vw_mean_annual", "vw_vol_annual", "corr_vw_ff",
            "ew_mean_annual", "ew_vol_annual", "top10_share", "MEMO"]
missing = [v for v in required if v not in globals()]
if missing:
    raise NameError(f"\n❌ Missing before submission: {missing}")

payload = {
    "assignment": "L2_Portfolios_AI",
    "ts": dt.datetime.now(dt.timezone.utc).isoformat(timespec="seconds").replace("+00:00", "Z"),
    "answers": {k: float(eval(k)) for k in required if k != "MEMO"},
    "memo": MEMO.strip(),
}
blob = json.dumps(payload, sort_keys=True)
checksum = hashlib.sha256(blob.encode()).hexdigest()[:8]
token = f"UG54::{checksum}::{base64.b64encode(blob.encode()).decode()}"

print("=" * 72)
print("📋  COPY THE LINE BELOW AND PASTE INTO THE SUBMISSION FORM")
print("=" * 72)
print(token)
print("=" * 72)
print(f"\nLength: {len(token)} chars")
print("Submission form: https://forms.gle/yazZ8bbatL87jdJi7")

---

## 🧠 Key Takeaways <a id="takeaways"></a>

1. **Stock data is stored long, one row per stock-month**, because the universe
   changes every month. `groupby('date')` is the operation you will use more
   than any other this semester.

2. **CRSP includes companies that failed.** Any dataset built from today's
   survivors has already answered your research question for you, wrongly.

3. **A portfolio is a weight vector.** Weights summing to 1 means fully
   invested; summing to 0 means self-financed long-short.

4. **A portfolio return is `w′r`** — a weighted average, one line of code.

5. **Weight by `me` at *t-1*, earn `ret` at *t*.** Pairing `me` at *t* with
   `ret` at *t* is look-ahead, and it is the most common way to accidentally
   invent a great strategy.

6. **Value-weighting is the market.** It's the only scheme all investors can
   hold simultaneously. Equal-weighting is an unintentional small-cap bet.

7. **Almost every stock has fatter tails than a normal** — 99% of them, average
   excess kurtosis 6.2. GE, at 0.5, is unusually well behaved; do not read one
   company as the rule.

8. **Stocks are skewed right, the market is skewed left.** A stock can rise
   2,400% and only fall 100%, so 89% lean right. The market leans left at −0.83,
   because crashes happen to everything at once. Diversification takes
   volatility from 52% to 15.5% and leaves the tail — excess kurtosis is still
   3.3, and the Sharpe ratio cannot see any of it.

9. **Over 1980–2000, equal-weighting lost by 1.5%/yr with more volatility.**
   The size premium died roughly when it was published — a theme we'll return
   to repeatedly.

---

### Next class

You can build *a* portfolio. Next: how to build one that's a **bet** — sorting
6,000 stocks on a signal and going long the top, short the bottom. Plus, at the
end, where this data really came from and how you'd pull it yourself.

---

## 📎 Appendix — What `groupby(...).apply(lambda g: ...)` Actually Does <a id="groupby"></a>

*Not lectured. Work through it if the one-liner in the demo went past you — and
if it did, you are in the majority.*

The line we ended up with was:

```python
vw = d.groupby('date').apply(lambda g: np.average(g['ret'], weights=g['me_l1']))
```

That is four unfamiliar things stacked into one expression: `groupby`, `.apply`,
an anonymous function, and `np.average` with a `weights=` argument. Each is
simple alone. Together they read as magic, and code that reads as magic is code
you cannot audit.

So we will compute the same number four ways, starting with a version that has
no magic in it at all, and check at the end that all four agree.

In [24]:
# The appendix is self-contained — rebuild the two columns we need.
panel['me_l1'] = panel.groupby('permno')['me'].shift(1)
d = panel.dropna(subset=['ret', 'me_l1'])

# ── 1 · ONE month, entirely by hand ───────────────────────────────────────
june = d[d.date == '1995-06-30']

total   = june['me_l1'].sum()      # total market value at the END of May
weights = june['me_l1'] / total    # each company's share of that total
rets    = june['ret']              # what each company returned during June

print(f"stocks alive in June 1995 : {len(june):,}")
print(f"weights sum to            : {weights.sum():.6f}")
print(f"market return, June 1995  : {(weights * rets).sum():.6f}")

stocks alive in June 1995 : 6,641
weights sum to            : 1.000000
market return, June 1995  : 0.031830


### The whole calculation is three lines

Nothing there is pandas-specific. You added up everyone's market value, divided
each company's value by that total to get its share, multiplied each share by
that company's return, and summed:

$$r_p = \sum_i w_i \, r_i, \qquad w_i = \frac{me_i}{\sum_j me_j}$$

The only problem is that you have to do it 251 times, once per month.

In [25]:
# ── 2 · EVERY month, with a plain for loop ────────────────────────────────
months  = sorted(d['date'].unique())
answers = {}

for m in months:                                  # 251 times round
    rows       = d[d.date == m]                   # just this month's stocks
    weights    = rows['me_l1'] / rows['me_l1'].sum()
    answers[m] = (weights * rows['ret']).sum()

by_loop = pd.Series(answers)
print(f"{len(by_loop)} monthly returns, computed one month at a time\n")
print(by_loop.head(3).round(6).to_string())

251 monthly returns, computed one month at a time

1980-02-29   -0.003410
1980-03-31   -0.116828
1980-04-30    0.051708


### This loop is correct, and it is the picture to keep in your head

Read it again. For each month: pull out that month's rows, compute the weights
*within* those rows, take the weighted sum, file it under that month's date.

Two things to notice, because both are the point of the lecture:

- **The weights are computed inside the loop.** They have to be — the universe
  changes every month, so June's denominator is not July's.
- **`me_l1` is *last* month's market value.** That is the only reason this is
  not look-ahead.

Everything from here is the same arithmetic written more briefly. If a shorter
version ever disagrees with this one, this one is right.

### `groupby` is that loop, written once

`d.groupby('date')` computes nothing. It splits the panel into pieces — one per
distinct date — and hands them to you. Iterate over it and you get exactly the
pairs the loop was building for itself:

In [27]:
# ── 3 · look inside a groupby ─────────────────────────────────────────────
for m, rows in d.groupby('date'):
    print(f"the key  `m`    is a {type(m).__name__}  ->  {m.date()}")
    print(f"the group `rows` is a {type(rows).__name__} with shape {rows.shape}\n")
    print(rows[['permno', 'date', 'ret', 'me_l1']].head(3).to_string(index=False))
                                  # just look at the first month and stop

the key  `m`    is a Timestamp  ->  1980-02-29
the group `rows` is a DataFrame with shape (4457, 9)

 permno       date       ret    me_l1
  10006 1980-02-29 -0.058795 367648.5
  10057 1980-02-29 -0.183582 142408.5
  10058 1980-02-29  0.000000   3127.5
the key  `m`    is a Timestamp  ->  1980-03-31
the group `rows` is a DataFrame with shape (4474, 9)

 permno       date       ret    me_l1
  10006 1980-03-31 -0.172078 341071.5
  10057 1980-03-31 -0.111111 114777.0
  10058 1980-03-31 -0.200000   3127.5
the key  `m`    is a Timestamp  ->  1980-04-30
the group `rows` is a DataFrame with shape (4482, 9)

 permno       date       ret      me_l1
  10006 1980-04-30 -0.027451 282380.625
  10057 1980-04-30 -0.141667 102384.000
  10058 1980-04-30  0.125000   2502.000
the key  `m`    is a Timestamp  ->  1980-05-31
the group `rows` is a DataFrame with shape (4484, 9)

 permno       date      ret     me_l1
  10006 1980-05-31 0.084677 274629.00
  10057 1980-05-31 0.185185  86386.50
  10058 1980-05-31

### `rows` is a DataFrame, not a row

That is the most common misreading, and the name `g` in the original one-liner
does nothing to help. Each piece is a **whole DataFrame** — every stock alive
that month, every column. It is the same object `d[d.date == m]` gave you in the
loop.

Once you know that, `.apply(f)` is easy to state: it runs `f` on each piece and
collects the answers into a Series indexed by the group key. So `f` has to be a
function that **takes a DataFrame and returns one number**.

In [28]:
# ── 4 · a NAMED function, applied to each group ───────────────────────────
def vw_return(rows):
    """Value-weighted return of one month's stocks."""
    weights = rows['me_l1'] / rows['me_l1'].sum()
    return (weights * rows['ret']).sum()

print(f"called directly on June 1995 : {vw_return(june):.6f}\n")

by_named = d.groupby('date').apply(vw_return)
print(f".apply(vw_return) returned {len(by_named)} numbers")
print(by_named.head(3).round(6).to_string())

called directly on June 1995 : 0.031830

.apply(vw_return) returned 251 numbers
date
1980-02-29   -0.003410
1980-03-31   -0.116828
1980-04-30    0.051708


### A `lambda` is that same function, without a name

These two are the same function:

```python
def vw_return(rows):
    return (rows['me_l1'] / rows['me_l1'].sum() * rows['ret']).sum()
```

```python
lambda rows: (rows['me_l1'] / rows['me_l1'].sum() * rows['ret']).sum()
```

`lambda` writes a one-expression function inline, where it is used, without
naming it first. The word right after `lambda` is just the parameter name —
`rows`, `g`, `x`, anything — and it gets bound to each group in turn. Whatever
the expression evaluates to is what gets returned.

That is the whole of it. There is no further machinery.

In [29]:
# ── 5 · the lambda versions, and all four compared ────────────────────────
by_lambda = d.groupby('date').apply(
    lambda rows: (rows['me_l1'] / rows['me_l1'].sum() * rows['ret']).sum())

# the one we actually use — np.average does the dividing for you
by_npavg = d.groupby('date').apply(
    lambda g: np.average(g['ret'], weights=g['me_l1']))

print(f"{'method':26s}{'months':>8s}{'mean/yr':>10s}{'max gap vs the loop':>22s}")
print("-" * 66)
for name, s in [('for loop',            by_loop),
                ('.apply(vw_return)',   by_named),
                ('.apply(lambda ...)',  by_lambda),
                ('.apply(np.average)',  by_npavg)]:
    gap = np.abs(s.values - by_loop.values).max()
    print(f"{name:26s}{len(s):>8}{s.mean()*12:>9.2%}{gap:>22.2e}")

method                      months   mean/yr   max gap vs the loop
------------------------------------------------------------------
for loop                       251   15.73%              0.00e+00
.apply(vw_return)              251   15.73%              0.00e+00
.apply(lambda ...)             251   15.73%              0.00e+00
.apply(np.average)             251   15.73%              1.49e-08


### Which one should you write?

All four are correct. Reach for them in this order:

1. **A named function** — when the calculation needs more than one line, when
   you will use it twice, or when a name would explain what it does. `vw_return`
   tells a reader what is happening; `lambda g:` does not.
2. **A lambda** — when the body is one short expression and a name would be
   noise.
3. **The loop** — when something disagrees and you want to stare at one month.

> **📌 The habit worth taking from this**
>
> When an AI hands you a line you cannot read aloud in English, do not run it
> and move on. Unstack it: a loop, then a named function, then the one-liner,
> and check that the answers match. It costs a few minutes, and it is how you
> find the version that is subtly wrong rather than obviously broken.

---

## 📎 Appendix — Data Loading <a id="appendix"></a>

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# 📎 APPENDIX — Belt-and-Suspenders Data Loading
# ═══════════════════════════════════════════════════════════════════════
# The panel is built once from CRSP and committed to the repo, so this notebook needs no WRDS account.
#
#     panel = pd.read_parquet("https://raw.githubusercontent.com/amoreira2/UG54/refs/heads/main/assets/data/panel_backbone_1980_2000.parquet")
#
# The build applies two corrections you should know about:
#   1. Delisting returns from crsp.msedelist are compounded into the monthly
#      return, so failed companies keep their final (often -100%) month.
#   2. The panel ships `ret` and `me` only. When you build a next-month return
#      from them, define it only where the next observation is the very next
#      calendar month, so a gap never silently splices across a missing period.


# ─── The panel: how it comes out of WRDS ───────────────────────────────
# You do not need to run this — the parquet is in the repo. It is here so you
# can see what the data looks like before anyone touched it, and so you can
# rebuild it over a different sample for your own project.
def pull_crsp_panel(start="1980-01-01", end="2000-12-31"):
    import wrds
    db = wrds.Connection()               # prompts for your WRDS credentials

    crsp = db.raw_sql(f"""
        select a.permno, a.date, a.ret, a.prc, a.shrout,
               b.exchcd, b.shrcd
        from crsp.msf a
        left join crsp.msenames b
          on  a.permno = b.permno
         and  b.namedt <= a.date
         and  a.date   <= b.nameendt
        where a.date between '{start}' and '{end}'
          and b.shrcd  in (10, 11)
          and b.exchcd in (1, 2, 3)
    """, date_cols=["date"])

    # Delisting returns. crsp.msf.ret is MISSING in the month a stock delists,
    # so a panel built without this loses the final (often -100%) month of every
    # company that failed, and every average you compute afterwards is biased
    # upward. Survivorship bias arriving as a missing value rather than a
    # missing row — nothing errors, the number is just wrong.
    dl = db.raw_sql(f"""
        select permno, dlstdt as date, dlret
        from crsp.msedelist
        where dlstdt between '{start}' and '{end}'
    """, date_cols=["date"])
    db.close()

    for df in (crsp, dl):                # align to month end so the keys match
        df["date"] = pd.to_datetime(df["date"]) + pd.offsets.MonthEnd(0)

    m = crsp.merge(dl, on=["permno", "date"], how="left")
    r, d = m["ret"], m["dlret"]
    m["ret"] = np.where(r.notna() & d.notna(), (1 + r) * (1 + d) - 1,
                np.where(r.isna() & d.notna(), d, r))

    # prc is NEGATIVE when there was no trade and CRSP stores the bid-ask
    # midpoint instead, so market equity takes the absolute value. The sign is
    # information — we use it in Lecture 3 — so prc is kept as delivered.
    m["me"] = m["prc"].abs() * m["shrout"]        # market equity, $thousands
    
    return (m.dropna(subset=["ret"])[["permno", "date", "ret", "me", "prc", "exchcd", "shrcd"]].sort_values(["permno", "date"]).reset_index(drop=True))

# ─── Fama-French factors: live fetch ───────────────────────────────────
# Prompt: "Using pandas-datareader, fetch F-F_Research_Data_Factors monthly from
#  the famafrench source starting 1980, convert the PeriodIndex to month-end
#  timestamps, and divide by 100 to get decimals."
def fetch_ff_monthly():
    import pandas_datareader.data as web
    f = web.DataReader('F-F_Research_Data_Factors', 'famafrench', start='1980-01-01')[0]
    f.index = pd.to_datetime(f.index.to_timestamp()) + pd.offsets.MonthEnd(0)
    return f / 100
